# week10 · 01 계면 인장 — 건조 · 습윤 셀 당기기

덱 5 · 14 · 21장. W08 빌더로 인장용 셀을 만들고(톨루엔 60분자 · 0.6 g/cm³) 그립을 0.0025 Å/fs 로 4,000 스텝 당긴다. 각 40초.

출력: `out/{dry,wet}/pull.csv` · `pull_summary.json`

실행(기본): 내 컴퓨터의 mdai · week10 폴더에서 Claude Code → 덱 14 · 21장 P10-1 · P10-2 지시(같은 스크립트를 직접 실행 · `../prompts.md`). 이 노트북 실행 = week10 폴더에서 `python ../run_nb.py notebooks/01_pull_test.ipynb`(작업 폴더 = week10 · nbconvert 는 경로가 어긋나 쓰지 않음) · 결과는 JupyterLab 으로 열어 본다. Colab 은 선택 경로.

In [ ]:
# 1) 폴더 확인 · 로컬 mdai(Claude Code · JupyterLab)이 기본 · 저장소가 없을 때(Colab 등 선택 경로)만 받기 · 이동
import os, subprocess, sys
if not os.path.exists('scripts/pull_test.py'):
    subprocess.run(['git','clone','https://github.com/sentirain7/2026_MOOC_Lecture_Materials.git'], check=False)
    os.chdir('2026_MOOC_Lecture_Materials/week10')
print(os.getcwd())

In [ ]:
# 2) LAMMPS · Packmol 확인 (mdai = environment.yml 에 포함 · W06 SETUP.md · Colab(선택)이면 condacolab 뒤 conda install -c conda-forge lammps packmol numpy)
# !conda install -y -c conda-forge lammps packmol numpy
!lmp -h | head -3 && packmol -h 2>&1 | head -1

In [ ]:
# 3) 건조 셀 인장 (약 40초)
!python3 scripts/pull_test.py --water 0 --tag dry --build --packmol packmol --lmp lmp

In [ ]:
# 4) 습윤 셀 인장 (물 10 Å · 약 50초)
!python3 scripts/pull_test.py --water 10 --tag wet --build --packmol packmol --lmp lmp

In [ ]:
# 5) 곡선 그리기 — σ(d) 두 조건
import csv, matplotlib.pyplot as plt
for tag, c in (('dry','#E37F1A'),('wet','#4F7FB0')):
    rows = list(csv.DictReader(open(f'out/{tag}/pull.csv')))
    d = [float(r['disp_A']) for r in rows]; s = [float(r['sigma_MPa']) for r in rows]
    plt.plot(d, s, color=c, label=tag)
plt.xlabel('d (Å)'); plt.ylabel('σ (MPa)'); plt.legend(); plt.show()

확인: 두 곡선이 피크 뒤 떨어지는가 · `pull_summary.json` 의 strain_rate_per_s 가 ~10¹¹ 인가(교육용). 바꿔 보기: `--v 0.005 --steps 2000 --tag dry_fast --cell dry`.